# Credit Card Fraud Detection MLOps Pipeline using TFX & Apache Beam
# Proyek: Membangun Machine Learning Pipeline Menggunakan Pipeline Orchestrator
- **Nama:** Muhammad Fathurrohman
- **ID Dicoding:** M_Fathurrohman

### Pipeline Orchestration:
Pipeline machine learning ini diorkestrasi secara menyeluruh menggunakan **Apache Beam (BeamDagRunner)**:
1. **Ingest Data** (`CsvExampleGen`)
2. **Compute Data Statistics** (`StatisticsGen`)
3. **Infer Schema** (`SchemaGen`)
4. **Validate Data** (`ExampleValidator`)
5. **Preprocess Data** (`Transform`)
6. **Tune Hyperparameters** (`Tuner`)
7. **Train Model** (`Trainer`)
8. **Resolve Baseline Model** (`Resolver`)
9. **Evaluate Model** (`Evaluator` via TFMA)
10. **Deploy / Serve Model** (`Pusher`)


## 1. Import Library and Dependencies

In [ ]:
import os
from absl import logging
import pandas as pd
import requests
import tensorflow as tf
import tensorflow_model_analysis as tfma
from keras_tuner import RandomSearch

# Importing TFX components
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher
)
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2

# TFX Pipeline Orchestrator using BeamDagRunner
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import pipeline
from tfx.orchestration import metadata

# Resolver for resolving inputs in pipelines
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy

# TFX standard artifact types
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

print("TensorFlow Version:", tf.__version__)
import tfx
print("TFX Version:", tfx.__version__)


## 2. Set Variables and Directory Configuration

In [ ]:
PIPELINE_NAME = "cc-fraud-pipeline"
MODEL_NAME = "cc-fraud-model"

# Directory for storing generated pipeline artifacts
PIPELINE_ROOT = os.path.join("cc_fraud_pipeline_artifacts", PIPELINE_NAME)

# Path to SQLite DB file for MLMD (Machine Learning Metadata) storage
METADATA_PATH = os.path.join("metadata", PIPELINE_NAME, "metadata.db")

# Output directory where trained model will be exported for serving
SERVING_MODEL_DIR = os.path.join("serving_model_dir", MODEL_NAME)

# Pipeline input paths and module files
DATA_ROOT = "cc_data"
TRANSFORM_MODULE_FILE = "modules/cc_fraud_transform.py"
TRAINER_MODULE_FILE = "modules/cc_fraud_trainer.py"
TUNER_MODULE_FILE = "modules/cc_fraud_tuner.py"

# Create directories if they do not exist
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs("modules", exist_ok=True)
os.makedirs(os.path.dirname(METADATA_PATH), exist_ok=True)

print("Pipeline Root:", PIPELINE_ROOT)
print("Metadata Path:", METADATA_PATH)
print("Serving Model Directory:", SERVING_MODEL_DIR)


## 3. Download and Prepare Dataset

In [ ]:
dataset_url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
csv_path = os.path.join(DATA_ROOT, "creditcard.csv")

if not os.path.exists(csv_path):
    print("Downloading Credit Card Fraud dataset...")
    response = requests.get(dataset_url, stream=True)
    response.raise_for_status()
    with open(csv_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("Download completed!")
else:
    print("Dataset already exists at:", csv_path)

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
print("Class distribution:")
print(df["Class"].value_counts())
df.head()


## 4. Define Transform Module (`cc_fraud_transform.py`)

In [ ]:
%%writefile {TRANSFORM_MODULE_FILE}
import tensorflow as tf
import tensorflow_transform as tft

# List of numerical features to scale
NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features by appending _xf"""
    return key + "_xf"

def preprocessing_fn(inputs):
    """tf.transform's callback function for preprocessing inputs."""
    outputs = {}
    
    # Standardize numerical features using Z-score scaling
    for key in NUMERICAL_FEATURES:
        outputs[transformed_name(key)] = tft.scale_to_z_score(inputs[key])
        
    # Pass through target label, casting to float32 for model compat
    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.float32)
    
    return outputs


## 5. Define Tuner Module (`cc_fraud_tuner.py`)

In [ ]:
%%writefile {TUNER_MODULE_FILE}
import os
import tensorflow as tf
import tensorflow_transform as tft
from tensorflow.keras import layers
from keras_tuner.engine import base_tuner
from keras_tuner import RandomSearch
import keras_tuner as kt
from tfx.components.trainer.fn_args_utils import FnArgs
from typing import Any, Dict, NamedTuple, Text

NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=None,
             batch_size=128) -> tf.data.Dataset:
    """Get post-transform features & create batches of data"""
    transform_feature_spec = tf_transform_output.transformed_feature_spec().copy()
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    
    return dataset

def model_builder(hp):
    """Build machine learning model for hyperparameter tuning"""
    num_layers = hp.Int('num_layers', min_value=1, max_value=3, step=1)
    dense_units = hp.Int('dense_units', min_value=32, max_value=128, step=32)
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4])

    inputs = {}
    for key in NUMERICAL_FEATURES:
        inputs[transformed_name(key)] = tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32)

    x = tf.keras.layers.concatenate(list(inputs.values()))

    for _ in range(num_layers):
        x = layers.Dense(dense_units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model

TunerFnResult = NamedTuple('TunerFnResult', [
    ('tuner', base_tuner.BaseTuner),
    ('fit_kwargs', Dict[Text, Any]),
])

def tuner_fn(fn_args: FnArgs):
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    train_steps = fn_args.train_steps if fn_args.train_steps and fn_args.train_steps > 0 else None
    eval_steps = fn_args.eval_steps if fn_args.eval_steps and fn_args.eval_steps > 0 else None

    train_set = input_fn(
        fn_args.train_files[0],
        tf_transform_output,
        num_epochs=None if train_steps else 1,
        batch_size=128
    )
    val_set = input_fn(
        fn_args.eval_files[0],
        tf_transform_output,
        num_epochs=None if eval_steps else 1,
        batch_size=128
    )

    model_tuner = RandomSearch(
        hypermodel=model_builder,
        objective=kt.Objective('val_accuracy', direction='max'),
        max_trials=3,
        executions_per_trial=1,
        directory=fn_args.working_dir,
        project_name='cc_fraud_tuner',
    )

    return TunerFnResult(
        tuner=model_tuner,
        fit_kwargs={
            'x': train_set,
            'validation_data': val_set,
            'steps_per_epoch': train_steps,
            'validation_steps': eval_steps,
            'callbacks': [
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy',
                    mode='max',
                    patience=2,
                    verbose=1
                )
            ]
        }
    )


## 6. Define Trainer Module (`cc_fraud_trainer.py`)

In [ ]:
%%writefile {TRAINER_MODULE_FILE}
import os
import tensorflow as tf
import tensorflow_transform as tft
from tensorflow.keras import layers
from tfx.components.trainer.fn_args_utils import FnArgs

NUMERICAL_FEATURES = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]
LABEL_KEY = "Class"

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=None,
             batch_size=128) -> tf.data.Dataset:
    """Get post-transform features & create batches of data"""
    transform_feature_spec = tf_transform_output.transformed_feature_spec().copy()
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    
    return dataset

def model_builder(hp):
    """Build machine learning model using hyperparameters"""
    num_layers = hp.get('num_layers', 2)
    dense_units = hp.get('dense_units', 64)
    dropout_rate = hp.get('dropout_rate', 0.2)
    learning_rate = hp.get('learning_rate', 1e-3)

    inputs = {}
    for key in NUMERICAL_FEATURES:
        inputs[transformed_name(key)] = tf.keras.Input(shape=(1,), name=transformed_name(key), dtype=tf.float32)

    x = tf.keras.layers.concatenate(list(inputs.values()))

    for _ in range(num_layers):
        x = layers.Dense(dense_units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model

def _get_serve_tf_examples_fn(model, tf_transform_output):
    """Returns a function that parses raw TF.Examples and runs model inference."""
    model.tft_layer = tf_transform_output.transform_features_layer()
    
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY)
        
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        
        return model(transformed_features)
        
    return serve_tf_examples_fn

def run_fn(fn_args: FnArgs) -> None:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)

    train_steps = fn_args.train_steps if fn_args.train_steps and fn_args.train_steps > 0 else None
    eval_steps = fn_args.eval_steps if fn_args.eval_steps and fn_args.eval_steps > 0 else None

    train_set = input_fn(
        fn_args.train_files,
        tf_transform_output,
        num_epochs=None if train_steps else 1,
        batch_size=128
    )
    val_set = input_fn(
        fn_args.eval_files,
        tf_transform_output,
        num_epochs=None if eval_steps else 1,
        batch_size=128
    )

    if fn_args.hyperparameters and 'values' in fn_args.hyperparameters:
        hp = fn_args.hyperparameters['values']
    else: 
        hp = {
            'num_layers': 2,
            'dense_units': 64,
            'dropout_rate': 0.2,
            'learning_rate': 1e-3
        }

    model = model_builder(hp)
    model.summary()

    log_dir = os.path.join(os.path.dirname(fn_args.serving_model_dir), 'logs')
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, update_freq='batch')
    es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', verbose=1, patience=5)
    
    checkpoint_dir = os.path.join(fn_args.serving_model_dir, 'checkpoint')
    mc = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, 'best_weights'),
        monitor='val_accuracy',
        mode='max',
        verbose=1,
        save_best_only=True,
        save_weights_only=True
    )

    model.fit(
        x=train_set,
        validation_data=val_set,
        epochs=10,
        steps_per_epoch=train_steps,
        validation_steps=eval_steps,
        callbacks=[tensorboard_callback, es, mc]
    )

    try:
        model.load_weights(os.path.join(checkpoint_dir, 'best_weights'))
        print("Successfully loaded best weights from checkpoint.")
    except Exception as e:
        print(f"Could not load best weights from checkpoint: {e}. Saving final epoch model.")

    signatures = {
        'serving_default': _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
            tf.TensorSpec(shape=[None], dtype=tf.string, name='examples')
        )
    }

    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)


## 7. Function to Initialize Pipeline Components (`init_components`)

In [ ]:
def init_components(
    data_dir: str,
    transform_module: str,
    training_module: str,
    tuner_module: str,
    training_steps: int = 500,
    eval_steps: int = 100,
    serving_model_dir: str = SERVING_MODEL_DIR,
):
    """Initializes all TFX components for Credit Card Fraud Detection Pipeline."""
    
    # 1. CsvExampleGen: Ingest dataset and split 80% train, 20% eval
    output = example_gen_pb2.Output(
        split_config=example_gen_pb2.SplitConfig(splits=[
            example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2)
        ])
    )
    example_gen = CsvExampleGen(input_base=data_dir, output_config=output)
    
    # 2. StatisticsGen: Compute data statistics
    statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
    
    # 3. SchemaGen: Infer data schema
    schema_gen = SchemaGen(
        statistics=statistics_gen.outputs['statistics'],
        infer_feature_shape=True
    )
    
    # 4. ExampleValidator: Validate data against schema
    example_validator = ExampleValidator(
        statistics=statistics_gen.outputs['statistics'],
        schema=schema_gen.outputs['schema']
    )
    
    # 5. Transform: Preprocessing using tf.transform
    transform = Transform(
        examples=example_gen.outputs['examples'],
        schema=schema_gen.outputs['schema'],
        module_file=os.path.abspath(transform_module)
    )
    
    # 6. Tuner: Hyperparameter tuning via KerasTuner
    tuner = Tuner(
        module_file=os.path.abspath(tuner_module),
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=training_steps),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=eval_steps)
    )
    
    # 7. Trainer: Train DNN model using best hyperparameters
    trainer = Trainer(
        module_file=os.path.abspath(training_module),
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        hyperparameters=tuner.outputs['best_hyperparameters'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=training_steps),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=eval_steps)
    )
    
    # 8. Resolver: Resolve baseline blessed model
    model_resolver = Resolver(
        strategy_class=LatestBlessedModelStrategy,
        model=Channel(type=Model),
        model_blessing=Channel(type=ModelBlessing)
    ).with_id('Latest_blessed_model_resolver')
    
    # 9. Evaluator: Evaluate candidate model vs baseline using TFMA
    eval_config = tfma.EvalConfig(
        model_specs=[tfma.ModelSpec(label_key='Class')],
        slicing_specs=[tfma.SlicingSpec()],
        metrics_specs=[
            tfma.MetricsSpec(metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(class_name='AUC'),
                tfma.MetricConfig(class_name='FalsePositives'),
                tfma.MetricConfig(class_name='TruePositives'),
                tfma.MetricConfig(class_name='FalseNegatives'),
                tfma.MetricConfig(class_name='TrueNegatives'),
                tfma.MetricConfig(class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.5}),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': 0.0001})
                    )
                )
            ])
        ]
    )
    
    evaluator = Evaluator(
        examples=example_gen.outputs['examples'],
        model=trainer.outputs['model'],
        baseline_model=model_resolver.outputs['model'],
        eval_config=eval_config
    )
    
    # 10. Pusher: Export blessed model for serving
    pusher = Pusher(
        model=trainer.outputs['model'],
        model_blessing=evaluator.outputs['blessing'],
        push_destination=pusher_pb2.PushDestination(
            filesystem=pusher_pb2.PushDestination.Filesystem(
                base_directory=serving_model_dir
            )
        )
    )
    
    components = (
        example_gen,
        statistics_gen,
        schema_gen,
        example_validator,
        transform,
        tuner,
        trainer,
        model_resolver,
        evaluator,
        pusher
    )
    
    return components


## 8. Function to Initialize Local Pipeline (`init_local_pipeline`)

In [ ]:
def init_local_pipeline(
    components,
    pipeline_root: str
) -> pipeline.Pipeline:
    """Initializes TFX Pipeline with ML Metadata and Apache Beam runner arguments."""
    
    logging.info(f"Pipeline root set to: {pipeline_root}")
    beam_args = [
        '--direct_running_mode=multi_processing',
        # 0 means auto-detect based on the number of CPUs available during execution time.
        '--direct_num_workers=0' 
    ]
    
    return pipeline.Pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=pipeline_root,
        components=components,
        enable_cache=True,
        metadata_connection_config=metadata.sqlite_metadata_connection_config(
            METADATA_PATH
        ),
        beam_pipeline_args=beam_args
    )


## 9. Run Pipeline using Apache Beam Orchestrator (`BeamDagRunner`)

In [ ]:
logging.set_verbosity(logging.INFO)

# 1. Inisialisasi seluruh komponen pipeline
components = init_components(
    data_dir=DATA_ROOT,
    transform_module=TRANSFORM_MODULE_FILE,
    training_module=TRAINER_MODULE_FILE,
    tuner_module=TUNER_MODULE_FILE,
    training_steps=500,
    eval_steps=100,
    serving_model_dir=SERVING_MODEL_DIR,
)

# 2. Inisialisasi pipeline lokal
tfx_pipeline = init_local_pipeline(components, PIPELINE_ROOT)

# 3. Jalankan pipeline menggunakan BeamDagRunner (Pipeline Orchestrator)
BeamDagRunner().run(pipeline=tfx_pipeline)


## 10. Verify Pipeline Execution & Exported Serving Model

In [ ]:
print("=== 1. Checking Metadata Database ===")
if os.path.exists(METADATA_PATH):
    print(f"ML Metadata SQLite DB exists: {METADATA_PATH} ({os.path.getsize(METADATA_PATH)} bytes)")
else:
    print(f"Metadata DB not found at: {METADATA_PATH}")

print("\n=== 2. Checking Exported Serving Model ===")
if os.path.exists(SERVING_MODEL_DIR):
    print(f"Serving model directory: {SERVING_MODEL_DIR}")
    for root, dirs, files in os.walk(SERVING_MODEL_DIR):
        print(f"{root} - {dirs} - {files}")
else:
    print("Model serving directory does not exist or Pusher was not triggered.")
